In [2]:

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

CSV_PATH = Path("classification_with_c110_d110_errors_snr.csv")

OUT_DIR = Path.cwd() / "svm_c110_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
REPORT_DIR = OUT_DIR / "analysis_report"
for d in [OUT_DIR, RUNS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload)
    payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX + 1))

N_REPEATS = 100
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

SVM_PARAMS = dict(
    kernel="rbf",
    C=2.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    cache_size=1024,
    tol=1e-3,
    random_state=0,
)

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(
        K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS,
        TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
        THR_GRID_SIZE=THR_GRID_SIZE,
        SVM_PARAMS=SVM_PARAMS,
    ),
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status": "ready"})

print("OUT_DIR:", OUT_DIR.resolve())

OUT_DIR: /Users/kris/Desktop/Astrophysics/svm_c110_truncation_out


In [3]:
df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain 'y'")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

# c000..c054 = BP, c055..c109 = RP
BP = [f"c{i:03d}" for i in range(55)]
RP = [f"c{i:03d}" for i in range(55, 110)]

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))

Xbp: (2815, 55) Xrp: (2815, 55) pos_rate: 0.19822380106571935


In [4]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(
            n_splits=1,
            test_size=VAL_FRAC_OF_REMAIN,
            random_state=base_seed + 1000 + r
        )
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})

splits: 100 sizes: {'train_idx': 1969, 'val_idx': 423, 'test_idx': 423}


In [5]:
def make_X_for_K(idx, K):
    bp = Xbp[idx, :K]   # (N,K)
    rp = Xrp[idx, :K]   # (N,K)
    X = np.concatenate([bp, rp], axis=1).astype(np.float32, copy=False)  # (N, 2K)
    return X

Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)

demo shape: (5, 20)


In [6]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if np.isfinite(prec) and np.isfinite(sens) and (prec+sens) > 0 else -np.inf
        else:
            raise ValueError(f"Unknown criterion: {criterion}")
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k, v in prob_metrics(yt, pt).items()}
    for crit in ["youden", "f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = float(thr)
        out[f"{crit}_val_score"] = float(sc)
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn)
        out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn)
        out[f"{crit}_test_tp"] = int(tp)
    return out

In [7]:
def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

print("Existing shards:", len(list(RUNS_DIR.glob("*.parquet"))))

Existing shards: 5500


In [8]:
pK = tqdm(K_LIST, desc="K (coeffs per BP/RP)", leave=True)

for K in pK:
    pK.set_postfix_str(f"K={K}")
    log_event(f"Starting K={K}")

    p_rep = tqdm(splits, desc=f"Repeats [K={K}]", leave=False)
    for sp in p_rep:
        rep = sp["repeat"]
        if done(K, rep):
            continue

        hb({"status": "run_start", "K": int(K), "repeat": int(rep)})

        tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

        Xtr = make_X_for_K(tr, K); ytr = y_all[tr]
        Xva = make_X_for_K(va, K); yva = y_all[va]
        Xte = make_X_for_K(te, K); yte = y_all[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xva = scaler.transform(Xva)
        Xte = scaler.transform(Xte)

        svm_params = dict(SVM_PARAMS)
        svm_params["random_state"] = BASE_SEED + rep

        model = SVC(**svm_params)

        t0 = time.time()
        model.fit(Xtr, ytr)

        pv = model.predict_proba(Xva)[:, 1]
        pt = model.predict_proba(Xte)[:, 1]

        met = evaluate(yva, pv, yte, pt)
        dt = time.time() - t0

        n_support_neg = int(model.n_support_[0]) if hasattr(model, "n_support_") else np.nan
        n_support_pos = int(model.n_support_[1]) if hasattr(model, "n_support_") else np.nan
        n_support_total = int(np.sum(model.n_support_)) if hasattr(model, "n_support_") else np.nan
        support_frac = float(n_support_total / len(ytr)) if np.isfinite(n_support_total) else np.nan

        row = dict(
            run_id=run_id(K, rep),
            K=int(K),
            repeat=int(rep),
            runtime_s=float(dt),
            n_train=int(len(ytr)),
            n_val=int(len(yva)),
            n_test=int(len(yte)),
            test_pos_rate=float(np.mean(yte)),
            n_support_neg=n_support_neg,
            n_support_pos=n_support_pos,
            n_support_total=n_support_total,
            support_frac=support_frac,
            **met
        )
        save_row(row)

        hb({"status": "run_done", "K": int(K), "repeat": int(rep), "runtime_s": dt})

log_event("ALL DONE.")
hb({"status": "completed"})
print("Done. Shards:", len(list(RUNS_DIR.glob('*.parquet'))))

K (coeffs per BP/RP):   0%|          | 0/55 [00:00<?, ?it/s]

Repeats [K=1]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=2]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=3]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=4]:   0%|          | 0/100 [00:00<?, ?it/s]

...

Repeats [K=52]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=53]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=54]:   0%|          | 0/100 [00:00<?, ?it/s]

Repeats [K=55]:   0%|          | 0/100 [00:00<?, ?it/s]

Done. Shards: 5500


In [9]:
res = load_all_rows()
print("rows:", res.shape)
display(res.head(5))

res.to_parquet(OUT_DIR / "truncation_svm_raw.parquet", index=False)
res.to_csv(OUT_DIR / "truncation_svm_raw.csv", index=False)
print("Saved raw outputs.")

rows: (5500, 34)


,run_id,K,repeat,runtime_s,n_train,n_val,n_test,test_pos_rate,n_support_neg,n_support_pos,...,youden_test_tp,f1_val_thr,f1_val_score,f1_test_Precision,f1_test_Sensitivity,f1_test_Specificity,f1_test_tn,f1_test_fp,f1_test_fn,f1_test_tp
0,K01__r00,1,0,0.424445,1969,423,423,0.198582,413,108,...,74,0.487742,0.813333,0.873418,0.821429,0.970501,329,10,15,69
1,K01__r01,1,1,0.408232,1969,423,423,0.198582,422,112,...,71,0.257385,0.846154,0.883117,0.809524,0.973451,330,9,16,68
2,K01__r02,1,2,0.402482,1969,423,423,0.198582,395,108,...,61,0.785135,0.860759,0.898305,0.630952,0.982301,333,6,31,53
3,K01__r03,1,3,0.416957,1969,423,423,0.198582,438,116,...,69,0.254802,0.860465,0.831325,0.821429,0.958702,325,14,15,69
4,K01__r04,1,4,0.411335,1969,423,423,0.198582,440,119,...,72,0.235406,0.860759,0.818182,0.857143,0.952802,323,16,12,72


Saved raw outputs.


In [10]:
metrics = [
    "runtime_s",
    "n_support_neg","n_support_pos","n_support_total","support_frac",
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Precision","youden_test_Sensitivity","youden_test_Specificity","youden_val_thr",
    "f1_test_Precision","f1_test_Sensitivity","f1_test_Specificity","f1_val_thr"
]

summ = res.groupby("K", as_index=False)[metrics].agg(["mean","std"])
summ.columns = [c if s == "" else f"{c}_{s}" for c, s in summ.columns]
summ = summ.sort_values("K")

summ.to_csv(OUT_DIR / "truncation_svm_summary_byK.csv", index=False)
display(summ.head(10))

,K,runtime_s_mean,runtime_s_std,n_support_neg_mean,n_support_neg_std,n_support_pos_mean,n_support_pos_std,n_support_total_mean,n_support_total_std,support_frac_mean,...,youden_val_thr_mean,youden_val_thr_std,f1_test_Precision_mean,f1_test_Precision_std,f1_test_Sensitivity_mean,f1_test_Sensitivity_std,f1_test_Specificity_mean,f1_test_Specificity_std,f1_val_thr_mean,f1_val_thr_std
0,1,0.418497,0.012635,418.34,18.044961,111.83,4.724864,530.17,22.559486,0.269259,...,0.195291,0.096603,0.872977,0.041690,0.790238,0.050046,0.970855,0.011369,0.373033,0.183934
1,2,0.422192,0.005330,397.54,17.770376,112.69,5.054641,510.23,22.471466,0.259132,...,0.242294,0.133282,0.890420,0.046071,0.791548,0.050152,0.975044,0.012469,0.493971,0.174234
2,3,0.427610,0.005785,383.32,17.236841,112.84,4.808914,496.16,21.508617,0.251986,...,0.242547,0.139871,0.886045,0.043958,0.807738,0.048079,0.973481,0.012155,0.459593,0.174561
3,4,0.426637,0.006590,393.00,16.991977,117.26,5.535232,510.26,21.757442,0.259147,...,0.256384,0.145322,0.884813,0.042622,0.809881,0.048408,0.973127,0.011961,0.431101,0.147464
4,5,0.428515,0.010021,404.01,17.752789,123.36,5.627350,527.37,22.416920,0.267836,...,0.288365,0.133315,0.886008,0.040393,0.806429,0.049292,0.973599,0.011112,0.427028,0.127900
5,6,0.431274,0.008199,411.75,18.483613,124.15,5.543902,535.90,23.076728,0.272169,...,0.259814,0.141070,0.883798,0.041991,0.809405,0.046999,0.972950,0.011688,0.404212,0.106413
6,7,0.443868,0.019854,418.49,17.986805,131.56,5.259566,550.05,22.211279,0.279355,...,0.246745,0.120414,0.876416,0.045552,0.807976,0.047789,0.970973,0.012839,0.387273,0.113760
7,8,0.436851,0.010506,426.39,17.845518,137.02,5.421581,563.41,22.011978,0.286140,...,0.245639,0.101212,0.859779,0.049330,0.810238,0.052093,0.966136,0.014647,0.362249,0.121103
8,9,0.438257,0.006886,435.02,17.130503,140.36,5.428172,575.38,21.360201,0.292219,...,0.233717,0.091692,0.861902,0.046831,0.800476,0.055378,0.967257,0.013763,0.367497,0.112064
9,10,0.441385,0.007142,446.01,18.306313,146.89,5.844612,592.90,22.809488,0.301117,...,0.228998,0.086762,0.857285,0.048434,0.804048,0.057441,0.965693,0.014079,0.355576,0.117245


In [11]:
import plotly.graph_objects as go
import plotly.express as px

def line_with_band(metric):
    dfm = res.groupby("K")[metric].agg(["mean","std"]).reset_index().sort_values("K")
    dfm.columns = ["K","mean","std"]
    dfm["lo"] = dfm["mean"] - dfm["std"]
    dfm["hi"] = dfm["mean"] + dfm["std"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["mean"], mode="lines+markers", name=f"{metric} mean"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["hi"], mode="lines", line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(
        x=dfm["K"], y=dfm["lo"], mode="lines", fill="tonexty",
        line=dict(width=0), name="±1 std", opacity=0.2
    ))
    fig.update_layout(
        title=f"{metric} vs K (mean ± std across repeats)",
        xaxis_title="K",
        yaxis_title=metric,
        height=520
    )
    fig.show()

for m in [
    "test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
    "youden_test_Sensitivity","youden_test_Specificity","youden_test_Precision",
    "f1_test_Sensitivity","f1_test_Specificity","f1_test_Precision"
]:
    line_with_band(m)